In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, ArrayType

# Create Spark session
spark = SparkSession.builder \
    .appName("Spark with Hive") \
    .enableHiveSupport() \
    .getOrCreate()

26/05/03 07:18:41 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# Define the schema
schema_campaigns = StructType([
    StructField("campaign_id", StringType(), True),
    StructField("campaign_name", StringType(), True),
    StructField("campaign_country", StringType(), True),
    StructField("os_type", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("place_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_time", TimestampType(), True)
])


schema_users = StructType([
    StructField("user_id", StringType(), True),
    StructField("country", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age_group", StringType(), True),
    StructField("category", ArrayType(StringType()), True)
])


schema_stores = StructType([
    StructField("store_name", StringType(), True),
    StructField("place_ids", ArrayType(StringType()), True)
])

In [3]:
# # Load the JSON data
hdfs_path1 = '/tmp/data/ad_campaigns_data.json'
hdfs_path2 = '/tmp/data/user_profile_data.json'
hdfs_path3 = '/tmp/data/store_data.json'

df_campaigns = spark.read.format('json').option("multiline", "true").schema(schema_campaigns).load(hdfs_path1)
df_users = spark.read.format('json').option("multiline", "true").schema(schema_users).load(hdfs_path2)
df_stores = spark.read.format('json').option("multiline", "true").schema(schema_stores).load(hdfs_path3)

In [6]:
df_campaigns.show()
df_users.show()
df_stores.show()

+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+
|campaign_id|       campaign_name|campaign_country|os_type|device_type| place_id|            user_id|event_type|         event_time|
+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+
|    ABCDFAE|Food category tar...|             USA|    ios|      apple|CASSBB-11|1264374214654454321|impression|2018-10-12 13:10:05|
|    ABCDFAE|Food category tar...|             USA|android|   MOTOROLA|CADGBD-13|1674374214654454321|impression|2018-10-12 13:09:04|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|BADGBA-12|   5747421465445443|  video ad|2018-10-12 13:10:10|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|CASSBB-11|1864374214654454132|     click|2018-10-12 13:10:12|
+-----------+--------------------+----------------+-------+----------

+-------------------+-------+------+---------+--------------------+
|            user_id|country|gender|age_group|            category|
+-------------------+-------+------+---------+--------------------+
|1264374214654454321|    USA|  male|    18-25|  [shopper, student]|
|1674374214654454321|    USA|female|    25-50|            [parent]|
|   5747421465445443|    USA|  male|    25-50|[shopper, parent,...|
|1864374214654454132|    USA|  male|      50+|      [professional]|
|  14537421465445443|    USA|female|    18-25|  [shopper, student]|
|  25547421465445443|    USA|female|      50+|[shopper, profess...|
+-------------------+-------+------+---------+--------------------+



+-------------+--------------------+
|   store_name|           place_ids|
+-------------+--------------------+
|     McDonald|[CASSBB-11, CADGB...|
|   BurgerKing|         [CASSBB-11]|
|        Macys|[BADGBA-13, CASSB...|
|shoppers stop|         [BADGBA-12]|
+-------------+--------------------+



In [7]:
df_campaigns = df_campaigns.withColumn("event_time",F.col("event_time").cast("timestamp"))
df_campaigns = df_campaigns.withColumn("date",F.to_date("event_time"))
df_campaigns = df_campaigns.withColumn("hour",F.hour("event_time"))

In [8]:
df_campaigns.show()

+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+
|campaign_id|       campaign_name|campaign_country|os_type|device_type| place_id|            user_id|event_type|         event_time|      date|hour|
+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+
|    ABCDFAE|Food category tar...|             USA|    ios|      apple|CASSBB-11|1264374214654454321|impression|2018-10-12 13:10:05|2018-10-12|  13|
|    ABCDFAE|Food category tar...|             USA|android|   MOTOROLA|CADGBD-13|1674374214654454321|impression|2018-10-12 13:09:04|2018-10-12|  13|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|BADGBA-12|   5747421465445443|  video ad|2018-10-12 13:10:10|2018-10-12|  13|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|CASSBB-11|1864374214654454132|     

In [11]:
re = (df_campaigns.groupBy("campaign_id", "date", "hour", "os_type", "event_type"))
re.show()

AttributeError: 'GroupedData' object has no attribute 'show'

In [15]:
re = ( 
    df_campaigns.groupBy("campaign_id", "date", "hour", "os_type", "event_type")
    .agg(F.count("event_type").alias("event_count"))
   )

In [16]:
re.show()

+-----------+----------+----+-------+----------+-----------+
|campaign_id|      date|hour|os_type|event_type|event_count|
+-----------+----------+----+-------+----------+-----------+
|    ABCDFAE|2018-10-12|  13|android|  video ad|          1|
|    ABCDFAE|2018-10-12|  13|android|impression|          1|
|    ABCDFAE|2018-10-12|  13|android|     click|          1|
|    ABCDFAE|2018-10-12|  13|    ios|impression|          1|
+-----------+----------+----+-------+----------+-----------+



In [17]:
result_adCampaign = (
    df_campaigns.groupBy("campaign_id", "date", "hour", "os_type", "event_type")
    .agg(F.count("event_type").alias("event_count"))
    .groupBy("campaign_id", "date", "hour", "os_type")
    .pivot("event_type")
    .agg(F.first("event_count"))
    .fillna(0)
    .select(
        "campaign_id",
        "date",
        "hour",
        "os_type",
        F.struct(
            F.col("impression").alias("impression"),
            F.col("click").alias("click"),
            F.col("video ad").alias("video_ad"),
        ).alias("event"),
    )
)

In [18]:
result_adCampaign.show()

+-----------+----------+----+-------+---------+
|campaign_id|      date|hour|os_type|    event|
+-----------+----------+----+-------+---------+
|    ABCDFAE|2018-10-12|  13|android|{1, 1, 1}|
|    ABCDFAE|2018-10-12|  13|    ios|{1, 0, 0}|
+-----------+----------+----+-------+---------+



In [22]:
hdfs_Outputpath = '/tmp/output/'
result_adCampaign.write.json(hdfs_Outputpath + "q1_output", mode="overwrite")

In [24]:
df_campaigns.show()
df_users.show()
df_stores.show()




+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+
|campaign_id|       campaign_name|campaign_country|os_type|device_type| place_id|            user_id|event_type|         event_time|      date|hour|
+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+
|    ABCDFAE|Food category tar...|             USA|    ios|      apple|CASSBB-11|1264374214654454321|impression|2018-10-12 13:10:05|2018-10-12|  13|
|    ABCDFAE|Food category tar...|             USA|android|   MOTOROLA|CADGBD-13|1674374214654454321|impression|2018-10-12 13:09:04|2018-10-12|  13|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|BADGBA-12|   5747421465445443|  video ad|2018-10-12 13:10:10|2018-10-12|  13|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|CASSBB-11|1864374214654454132|     

+-------------------+-------+------+---------+--------------------+
|            user_id|country|gender|age_group|            category|
+-------------------+-------+------+---------+--------------------+
|1264374214654454321|    USA|  male|    18-25|  [shopper, student]|
|1674374214654454321|    USA|female|    25-50|            [parent]|
|   5747421465445443|    USA|  male|    25-50|[shopper, parent,...|
|1864374214654454132|    USA|  male|      50+|      [professional]|
|  14537421465445443|    USA|female|    18-25|  [shopper, student]|
|  25547421465445443|    USA|female|      50+|[shopper, profess...|
+-------------------+-------+------+---------+--------------------+



+-------------+--------------------+
|   store_name|           place_ids|
+-------------+--------------------+
|     McDonald|[CASSBB-11, CADGB...|
|   BurgerKing|         [CASSBB-11]|
|        Macys|[BADGBA-13, CASSB...|
|shoppers stop|         [BADGBA-12]|
+-------------+--------------------+



In [28]:
result_store = (
     df_campaigns.join(df_stores, F.array_contains(df_stores.place_ids, df_campaigns.place_id), "inner")
    .groupBy("campaign_id","date","hour","store_name","event_type")
    .agg(F.count("event_type").alias("event_column"))
    .groupBy("campaign_id", "date", "hour", "store_name")
    .pivot("event_type")
    .agg(F.first("event_column"))
    .fillna(0)
    .select (
        "campaign_id",
        "date",
        "hour",
        "store_name",
        F.struct(
            F.col("impression").alias("impression"),
            F.col("click").alias("click"),
            F.col("video ad").alias("video_ad"),
        ).alias("event"),
    )
)


In [29]:
result_store.show()

+-----------+----------+----+-------------+---------+
|campaign_id|      date|hour|   store_name|    event|
+-----------+----------+----+-------------+---------+
|    ABCDFAE|2018-10-12|  13|     McDonald|{2, 1, 0}|
|    ABCDFAE|2018-10-12|  13|shoppers stop|{0, 0, 1}|
|    ABCDFAE|2018-10-12|  13|   BurgerKing|{1, 1, 0}|
+-----------+----------+----+-------------+---------+



In [30]:
result_store.write.json(hdfs_Outputpath + "q2_output", mode="overwrite")

In [33]:
result_user = (
     df_campaigns.join(df_users,"user_id" , "inner")
    .groupBy("campaign_id","date","hour","gender","event_type")
    .agg(F.count("event_type").alias("event_column"))
    .groupBy("campaign_id", "date", "hour", "gender")
    .pivot("event_type")
    .agg(F.first("event_column"))
    .fillna(0)
    .select (
        "campaign_id",
        "date",
        "hour",
        "gender",
        F.struct(
            F.col("impression").alias("impression"),
            F.col("click").alias("click"),
            F.col("video ad").alias("video_ad"),
        ).alias("event"),
    )
)


In [34]:
result_user.show()

+-----------+----------+----+------+---------+
|campaign_id|      date|hour|gender|    event|
+-----------+----------+----+------+---------+
|    ABCDFAE|2018-10-12|  13|  male|{1, 1, 1}|
|    ABCDFAE|2018-10-12|  13|female|{1, 0, 0}|
+-----------+----------+----+------+---------+



In [35]:
result_user.write.json(hdfs_Outputpath + "q3_output", mode="overwrite")